In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
!pip install dspy-ai

In [ ]:
import threading
import subprocess

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()

In [ ]:
import dspy
import requests

In [ ]:
def setup_ollama():
  try:
    requests.get("http://localhost:11434/api/tags")
    print("Ollama is running")
  except:
    print("Ollama not running. Start with: ollama serve")
    return False
setup_ollama()

In [ ]:
!ollama pull llama3.2

In [ ]:
dspy.settings.configure(
    lm=dspy.LM(
      model="ollama/llama3.2",
      api_base="http://localhost:11434",
      max_tokens=500,
      temperature=0.7,
    )
)

In [ ]:
import sqlite3

def create_ecommerce_database():
  conn = sqlite3.connect("ecommerce.db")
  c = conn.cursor()

  c.execute("""CREATE TABLE IF NOT EXISTS clientes (
      id INTEGER PRIMARY KEY, nome TEXT, email TEXT, data_cadastro TEXT)""")
      
  c.execute("""CREATE TABLE IF NOT EXISTS produtos (
      id INTEGER PRIMARY KEY, nome TEXT, categoria TEXT, preco REAL, estoque INTEGER)""")
      
  c.execute("""CREATE TABLE IF NOT EXISTS pedidos (
      id INTEGER PRIMARY KEY, cliente_id INTEGER, produto_id INTEGER, quantidade INTEGER, status TEXT)""")

  c.executemany("INSERT OR IGNORE INTO clientes VALUES (?, ?, ?, ?)", [
    (1, "João Silva", "joao@email.com", "2023-01-15"),
    (2, "Maria Souza", "maria@email.com", "2023-05-20"),
    (3, "Carlos Oliveira", "carlos@email.com", "2023-08-10")
  ])

  c.executemany("INSERT OR IGNORE INTO produtos VALUES (?, ?, ?, ?, ?)", [
    (1, "Notebook Dell", "Eletrônicos", 3500.00, 10),
    (2, "Smartphone Samsung", "Eletrônicos", 1500.00, 25),
    (3, "Cadeira Gamer", "Móveis", 800.00, 5)
  ])

  c.executemany("INSERT OR IGNORE INTO pedidos VALUES (?, ?, ?, ?, ?)", [
    (1, 1, 1, 1, "Entregue"),
    (2, 2, 2, 2, "Enviado"),
    (3, 1, 3, 1, "Pendente")
  ])

  conn.commit()
  conn.close()
  print("Database Ecommerce criado")

create_ecommerce_database()

In [ ]:
class GenerateSQL(dspy.Signature):
  """Generate SQL from natural language. Respond ONLY with the reasoning and the valid SQL query. Do not add any conversational text or formatting.

  Database schema:
  - clientes: id, nome, email, data_cadastro
  - produtos: id, nome, categoria, preco, estoque
  - pedidos: id, cliente_id, produto_id, quantidade, status
  """

  question = dspy.InputField(desc="Natural language question")
  sql_query = dspy.OutputField(desc="Valid SQL query")

In [ ]:
class SQLGenerator(dspy.Module):
  def __init__(self):
    super().__init__()
    self.generator = dspy.ChainOfThought(GenerateSQL)
    self.refiner = dspy.ChainOfThought(
      "question, sql_query, error -> refined_sql"
    )

  def forward(self, question, conn):
    output = self.generator(question=question)
    sql = output.sql_query.strip()
    
    # Remove markdown caso o modelo gere
    sql = sql.replace("```sql", "").replace("```", "").strip()

    try:
      results = conn.execute(sql).fetchall()
      return dspy.Prediction(sql_query=sql, results=results, error=None)
    except Exception as e:
      refined = self.refiner(question=question, sql_query=sql, error=str(e))
      refined_sql = refined.refined_sql.replace("```sql", "").replace("```", "").strip()
      try:
        results = conn.execute(refined_sql).fetchall()
        return dspy.Prediction(sql_query=refined_sql, results=results, error=None)
      except Exception as e2:
        return dspy.Prediction(sql_query=refined_sql, results=None, error=str(e2))

In [ ]:
def create_examples():
  return [
    dspy.Example(
      question="Quantos produtos temos em estoque na categoria Eletrônicos?",
      reasoning="Preciso somar a coluna estoque filtrando apenas a categoria Eletrônicos.",
      sql_query="SELECT SUM(estoque) FROM produtos WHERE categoria = 'Eletrônicos'"
    ).with_inputs("question"),
    dspy.Example(
      question="Qual o nome do cliente que fez o pedido número 1?",
      reasoning="Preciso fazer um JOIN entre clientes e pedidos para achar o nome correspondente ao pedido 1.",
      sql_query="SELECT c.nome FROM clientes c JOIN pedidos p ON c.id = p.cliente_id WHERE p.id = 1"
    ).with_inputs("question"),
    dspy.Example(
      question="Qual é o produto mais barato?",
      reasoning="Devo ordenar os produtos pelo preço em ordem crescente e pegar o primeiro.",
      sql_query="SELECT nome, preco FROM produtos ORDER BY preco ASC LIMIT 1"
    ).with_inputs("question")
  ]

generator = SQLGenerator()
generator.generator.demos = create_examples()
print("✓ Generator configured with few-shot examples")

In [ ]:
conn = sqlite3.connect("ecommerce.db")
test_questions = [
    "Quantos clientes estão cadastrados?",
    "Qual o produto mais caro?",
    "Liste os pedidos que estão com status Pendente",
    "Qual a quantidade total de smartphones vendidos?",
]

for i, question in enumerate(test_questions, 1):
    print(f"Query {i}: {question}")
    result = generator(question=question, conn=conn)
    print(f"SQL: {result.sql_query}")
    print(f"Results: {result.results if not result.error else f' {result.error}'}\n")